In [47]:

"""
CrawlPrioritizer: PageRank-aware URL prioritisation for AI training data crawling.

============================================================
DESIGN RATIONALE
============================================================

When building training corpora for large language models (GPTBot-style),
not all web pages are equally valuable.  High-PageRank pages tend to be:

  1. Authoritative  -- many trusted sites link to them.
  2. Information-dense -- well-curated hubs (Wikipedia, arXiv, official docs).
  3. Low-noise  -- spammy pages rarely accumulate many quality backlinks.

We combine three signals into a composite PRIORITY SCORE:

    priority(u) = w_pr * norm(PR(u))                  (PageRank authority)
               + w_out * norm(out_degree(u))           (hub potential: many links to follow)
               + w_robots * robots_bonus(u)            (page respects/signals crawlability)

All components are normalised to [0, 1] before weighting so the weights
are interpretable percentages.

Heuristic for "high quality AND permits crawling":
  - Download robots.txt for each domain.
  - Pages whose domain does NOT block GPTBot/CCBot/OAI-SearchBot receive a
    positive robots_bonus.
  - This surfaces authoritative pages that actively welcome AI crawlers,
    which are more likely to contain consent-cleared training data.
============================================================
"""

from __future__ import annotations

import heapq
import logging
import re
import urllib.parse
from dataclasses import dataclass, field
from typing import Optional
import time

import numpy as np

logger = logging.getLogger(__name__)

# Known AI crawler user-agent tokens to check in robots.txt
_AI_BOTS = ["GPTBot", "CCBot", "OAI-SearchBot", "anthropic-ai", "Googlebot"]
_ROBOTS_DISALLOW_RE = re.compile(r"^Disallow\s*:\s*(.+)$", re.IGNORECASE)
_USERAGENT_RE = re.compile(r"^User-agent\s*:\s*(.+)$", re.IGNORECASE)


@dataclass(order=True)
class CrawlCandidate:
    """A URL candidate with its composite crawl-priority score."""

    priority: float = field(compare=True)  # negated for min-heap use
    url: str = field(compare=False)
    pagerank: float = field(compare=False)
    out_degree: int = field(compare=False)
    robots_ok: bool = field(compare=False)
    reason: str = field(compare=False, default="")


class CrawlPrioritizer:
    """
    Rank a set of discovered URLs by crawl priority for AI training data.

    Parameters
    ----------
    graph       : dict[str, list[str]]  — adjacency dict  {url: [outlink_urls]}
    pageranks   : dict[str, float]      — precomputed PageRank scores
    w_pr        : weight for PageRank signal   (default 0.6)
    w_out       : weight for out-degree signal (default 0.2)
    w_robots    : weight for robots.txt signal (default 0.2)
    check_robots: whether to fetch robots.txt files (default False; requires network)
    """

    def __init__(
        self,
        graph: dict[str, list[str]],
        pageranks: dict[str, float],
        w_pr: float = 0.6,
        w_out: float = 0.2,
        w_robots: float = 0.2,
        check_robots: bool = False,
    ) -> None:
        self.graph = graph
        self.pageranks = pageranks
        self.w_pr = w_pr
        self.w_out = w_out
        self.w_robots = w_robots
        self.check_robots = check_robots
        self._robots_cache: dict[str, bool] = {}  # domain -> True if crawlable

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def top_k(self, k: int = 10) -> list[CrawlCandidate]:
        """Return the top-k URLs to crawl next, ranked by composite priority."""
        urls = list(self.graph.keys())
        if not urls:
            return []

        pr_arr = np.array([self.pageranks.get(u, 0.0) for u in urls], dtype=float)
        print(pr_arr)
        od_arr = np.array([len(self.graph.get(u, [])) for u in urls], dtype=float)
        print(od_arr)

        pr_norm = self._normalise(pr_arr)
        print(pr_norm)
        od_norm = self._normalise(od_arr)
        print(od_norm)

        # Build robots signal
        robots_arr = np.zeros(len(urls))
        if self.check_robots:
            for i, url in enumerate(urls):
                robots_arr[i] = 1.0 if self._is_crawlable(url) else 0.0
        else:
            robots_arr[:] = 1.0  # assume all crawlable if not checking
        print(robots_arr)

        priority_arr = (
            self.w_pr * pr_norm
            + self.w_out * od_norm
            + self.w_robots * robots_arr
        )
        print(priority_arr)

        # Use a heap to get top-k efficiently (O(N log k))
        heap: list[tuple[float, int]] = []
        for i, score in enumerate(priority_arr):
            if len(heap) < k:
                heapq.heappush(heap, (score, i))
            elif score > heap[0][0]:
                heapq.heapreplace(heap, (score, i))

        top_indices = sorted([i for _, i in heap], key=lambda i: -priority_arr[i])

        candidates = []
        for i in top_indices:
            url = urls[i]
            reason = self._explain(pr_norm[i], od_norm[i], robots_arr[i])
            candidates.append(
                CrawlCandidate(
                    priority=float(priority_arr[i]),
                    url=url,
                    pagerank=float(pr_arr[i]),
                    out_degree=int(od_arr[i]),
                    robots_ok=bool(robots_arr[i] > 0),
                    reason=reason,
                )
            )
        return candidates

    def explain_policy(self) -> str:
        """Return a human-readable explanation of the crawl policy."""
        return (
            "Crawl Priority Policy\n"
            "=====================\n"
            f"  PageRank weight  : {self.w_pr:.0%}  — authoritative pages first\n"
            f"  Out-degree weight: {self.w_out:.0%}  — hubs surface more new URLs\n"
            f"  Robots bonus     : {self.w_robots:.0%}  — prefer consent-cleared pages\n\n"
            "Why high-PageRank pages yield better AI training data:\n"
            "  PageRank is a proxy for editorial endorsement — many trusted\n"
            "  sites must have found a page worth linking to.  High-PR pages\n"
            "  tend to be factually accurate, well-written, and frequently\n"
            "  updated (e.g. Wikipedia, arXiv, official documentation).\n"
            "  These properties directly correlate with training data quality\n"
            "  for generative models.\n\n"
            "Robots.txt heuristic:\n"
            "  Pages whose domain does NOT block known AI bots (GPTBot,\n"
            "  CCBot, anthropic-ai) in robots.txt are more likely to have\n"
            "  consented to AI training use.  Combining PageRank authority\n"
            "  with robots-crawlability surfaces high-quality, consent-cleared\n"
            "  training pages."
        )

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _normalise(arr: np.ndarray) -> np.ndarray:
        """Min-max normalise to [0, 1]; return zeros if all values identical."""
        lo, hi = arr.min(), arr.max()
        if hi == lo:
            return np.zeros_like(arr)
        return (arr - lo) / (hi - lo)

    def _is_crawlable(self, url: str) -> bool:
        """Check robots.txt for the domain. Cached per domain."""
        try:
            parsed = urllib.parse.urlparse(url)
            domain = f"{parsed.scheme}://{parsed.netloc}"
        except Exception:
            return True  # assume crawlable on parse error

        if domain in self._robots_cache:
            return self._robots_cache[domain]

        robots_url = f"{domain}/robots.txt"
        try:
            import urllib.request
            with urllib.request.urlopen(robots_url, timeout=5) as resp:
                content = resp.read().decode("utf-8", errors="replace")
            result = self._parse_robots(content)
        except Exception:
            result = True  # network error → assume crawlable

        self._robots_cache[domain] = result
        return result

    @staticmethod
    def _parse_robots(content: str) -> bool:
        """
        Return True if none of the known AI bots are globally disallowed.

        Simple parser: looks for User-agent: <bot> followed by Disallow: /
        """
        lines = content.splitlines()
        current_agents: list[str] = []
        for line in lines:
            line = line.strip()
            ua_match = _USERAGENT_RE.match(line)
            if ua_match:
                current_agents = [ua_match.group(1).strip()]
                continue
            dis_match = _ROBOTS_DISALLOW_RE.match(line)
            if dis_match:
                path = dis_match.group(1).strip()
                if path in ("/", "*"):
                    for agent in current_agents:
                        for bot in _AI_BOTS:
                            if bot.lower() in agent.lower() or agent == "*":
                                return False  # globally disallowed
        return True

    @staticmethod
    def _explain(pr_norm: float, od_norm: float, robots_ok: float) -> str:
        parts = []
        if pr_norm > 0.7:
            parts.append("very high PageRank authority")
        elif pr_norm > 0.4:
            parts.append("moderate PageRank authority")
        if od_norm > 0.6:
            parts.append("high out-degree hub")
        if robots_ok:
            parts.append("robots.txt allows AI crawlers")
        return "; ".join(parts) if parts else "low priority"


In [48]:
# test_prioritizer.py  (run from project root: python test_prioritizer.py)

import sys
sys.path.insert(0, "src")

# ── 1. Toy graph ────────────────────────────────────────────────────────────
# 5 nodes, adjacency dict: {url: [outlinks]}
graph = {
    "https://wikipedia.org":  ["https://arxiv.org", "https://bbc.com"],
    "https://arxiv.org":      ["https://wikipedia.org"],
    "https://bbc.com":        ["https://wikipedia.org", "https://spam.io"],
    "https://spam.io":        [],                          # dangling node
    "https://docs.python.org":["https://wikipedia.org", "https://arxiv.org"],
}

# ── 2. Fake precomputed PageRank scores ─────────────────────────────────────
pageranks = {
    "https://wikipedia.org":   0.45,   # highest authority
    "https://arxiv.org":       0.25,
    "https://bbc.com":         0.15,
    "https://docs.python.org": 0.10,
    "https://spam.io":         0.05,   # lowest
}

# ── 3. Run prioritizer ──────────────────────────────────────────────────────
p = CrawlPrioritizer(
    graph=graph,
    pageranks=pageranks,
    w_pr=0.6,       # 60% PageRank
    w_out=0.2,      # 20% out-degree
    w_robots=0.2,   # 20% robots (check_robots=False → all get 1.0)
    check_robots=False,
)


In [49]:

# ── 4. Print top-k results ──────────────────────────────────────────────────
print("=== Top 5 URLs to crawl ===\n")
for rank, candidate in enumerate(p.top_k(k=5), 1):
    print(f"#{rank}  {candidate.url}")
    print(f"     priority  : {candidate.priority:.4f}")
    print(f"     pagerank  : {candidate.pagerank:.4f}")
    print(f"     out_degree: {candidate.out_degree}")
    print(f"     robots_ok : {candidate.robots_ok}")
    print(f"     reason    : {candidate.reason}")
    print()

# ── 5. Print crawl policy explanation ───────────────────────────────────────
print(p.explain_policy())


=== Top 5 URLs to crawl ===

[0.45 0.25 0.15 0.05 0.1 ]
[2. 1. 2. 0. 2.]
[1.    0.5   0.25  0.    0.125]
[1.  0.5 1.  0.  1. ]
[1. 1. 1. 1. 1.]
[1.    0.6   0.55  0.2   0.475]
#1  https://wikipedia.org
     priority  : 1.0000
     pagerank  : 0.4500
     out_degree: 2
     robots_ok : True
     reason    : very high PageRank authority; high out-degree hub; robots.txt allows AI crawlers

#2  https://arxiv.org
     priority  : 0.6000
     pagerank  : 0.2500
     out_degree: 1
     robots_ok : True
     reason    : moderate PageRank authority; robots.txt allows AI crawlers

#3  https://bbc.com
     priority  : 0.5500
     pagerank  : 0.1500
     out_degree: 2
     robots_ok : True
     reason    : high out-degree hub; robots.txt allows AI crawlers

#4  https://docs.python.org
     priority  : 0.4750
     pagerank  : 0.1000
     out_degree: 2
     robots_ok : True
     reason    : high out-degree hub; robots.txt allows AI crawlers

#5  https://spam.io
     priority  : 0.2000
     pagerank 

In [17]:
def _is_crawlable(url: str) -> bool:
    """Check robots.txt for the domain. Cached per domain."""
    try:
        parsed = urllib.parse.urlparse(url)
        print(parsed)
        domain = f"{parsed.scheme}://{parsed.netloc}"
        print(domain)
    except Exception:
        return True  # assume crawlable on parse error


    # robots_url = f"{domain}/robots.txt"
    # try:
    #     import urllib.request
    #     with urllib.request.urlopen(robots_url, timeout=5) as resp:
    #         content = resp.read().decode("utf-8", errors="replace")
    # except Exception:
    #     result = True  # network error → assume crawlable

    # return result

_is_crawlable("www.bbc.com")

ParseResult(scheme='', netloc='', path='www.bbc.com', params='', query='', fragment='')
://


In [22]:
url = "https://www.bbc.com/news/world-radio-and-tv-14563857?dbs"
parsed = urllib.parse.urlparse(url)
print(parsed)

ParseResult(scheme='https', netloc='www.bbc.com', path='/news/world-radio-and-tv-14563857', params='', query='dbs', fragment='')


In [23]:
domain = f"{parsed.scheme}://{parsed.netloc}"
domain

'https://www.bbc.com'

In [24]:
robots_url = f"{domain}/robots.txt"
import urllib.request
with urllib.request.urlopen(robots_url, timeout=5) as resp:
    content = resp.read().decode("utf-8", errors="replace")
content

"\n# version: 33da601c81953a45668bf5504fe0e8f8fb8aa953\n# The BBC's Terms of Use: https://www.bbc.co.uk/terms\n# - Explain the rules for using our services\n# - Tell you what you can do with our content\n#\n# In short: Please use our site like a human, not a robot.\n# That means:\n# - No scraping, crawling, or systematic extraction of content \n# - No use of BBC content for training or fine-tuning AI models, including large language models (LLMs)\n# - No retrieval-augmented generation (RAG), AI-powered search, agentic AI or grounding using BBC content\n# - No creating datasets from BBC content\n# - No text and data mining (TDM) under Article 4 of the EU Directive on Copyright in the Digital Single Market\n# - No using BBC content to create summaries for your own use\n# - No business use without permission (details: https://www.bbc.co.uk/usingthebbc/terms/can-i-use-bbc-content-for-my-business/)\n# - The BBC reserves all rights in its content and expressly opts out of any statutory excep

In [37]:
def _parse_robots(content: str) -> bool:
    """
    Return True if none of the known AI bots are globally disallowed.

    Simple parser: looks for User-agent: <bot> followed by Disallow: /
    """
    lines = content.splitlines()
    current_agents: list[str] = []
    for line in lines:
        line = line.strip()
        ua_match = _USERAGENT_RE.match(line)
        if ua_match:
            print(line)
            print(f"ua_match:{ua_match}")
            
            current_agents = [ua_match.group(1).strip()]
            continue
        dis_match = _ROBOTS_DISALLOW_RE.match(line)
        if dis_match:
            print(line)
            print(f"dis_match: {dis_match}")
            path = dis_match.group(1).strip()
            if path in ("/", "*"):
                for agent in current_agents:
                    for bot in _AI_BOTS:
                        if bot.lower() in agent.lower() or agent == "*":
                            return False  # globally disallowed
    return True


In [46]:
from collections import deque
import urllib.request
from html.parser import HTMLParser

class LinkParser(HTMLParser):
    """Extract all href links from a page."""
    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            for attr, val in attrs:
                if attr == "href" and val:
                    self.links.append(val)


def crawl(seed_url, max_pages=10):
    visited = set()
    queue = deque([seed_url])

    while queue and len(visited) < max_pages:

        # ── 1. Pick next URL ──────────────────────────────────
        url = queue.popleft()
        if url in visited:
            continue

        # ── 2. Fetch page ─────────────────────────────────────
        try:
            with urllib.request.urlopen(url, timeout=5) as resp:
                html = resp.read().decode("utf-8", errors="replace")
                print(html)
        except Exception as e:
            print(f"  FAILED: {url} — {e}")
            continue

        # ── 3. Mark visited ───────────────────────────────────
        visited.add(url)
        print(f"  Crawled: {url}")

        # ── 4. Extract links ──────────────────────────────────
        parser = LinkParser()
        parser.feed(html)
        
        for link in parser.links:
            full_url = urllib.parse.urljoin(url, link) 
            print(f"Full url {full_url}")
            if full_url.startswith("http") and link not in visited:
                queue.append(full_url)

    return visited


# Run
crawl("http://quotes.toscrape.com", max_pages=5)


<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div class="col-md-4">
                <p>
                
                    <a href="/login">Login</a>
                
                </p>
            </div>
        </div>
    

<div class="row">
    <div class="col-md-8">

    <div class="quote" itemscope itemtype="http://schema.org/CreativeWork">
        <span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
        <span>by <small class="auth

{'http://quotes.toscrape.com',
 'http://quotes.toscrape.com/',
 'http://quotes.toscrape.com/author/Albert-Einstein',
 'http://quotes.toscrape.com/login',
 'http://quotes.toscrape.com/tag/change/page/1/'}